In [0]:
from pyspark.sql.functions import *
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

In [0]:
# %sql
# create a new databframe with three columns ie area , no of rooms, price in databricks
from pyspark.sql import SparkSession

# Create Spark Session (already available in Databricks)
spark = SparkSession.builder.getOrCreate()

# Sample data
data = [
    (1200, 3, 5000000),
    (1500, 4, 6500000),
    (1800, 4, 7200000),
    (2200, 5, 9000000),
    (1000, 2, 4200000)
]

# Column names
columns = ["area", "no_of_rooms", "price"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Display DataFrame
df.show()

+----+-----------+-------+
|area|no_of_rooms|  price|
+----+-----------+-------+
|1200|          3|5000000|
|1500|          4|6500000|
|1800|          4|7200000|
|2200|          5|9000000|
|1000|          2|4200000|
+----+-----------+-------+



In [0]:
df = spark.createDataFrame([(1000,1,10000),(2000,2,200000),(3000,3,300000),(4000,4,400000),(5000,5,500000)],["area","no_of_rooms","price"])
df.show()

+----+-----------+------+
|area|no_of_rooms| price|
+----+-----------+------+
|1000|          1| 10000|
|2000|          2|200000|
|3000|          3|300000|
|4000|          4|400000|
|5000|          5|500000|
+----+-----------+------+



In [0]:
from pyspark.sql.functions import *
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator


In [0]:
vector_assembler=VectorAssembler(inputCols=["area","no_of_rooms"],outputCol="features")

In [0]:
updated_df=vector_assembler.transform(df)
updated_df.show()

+----+-----------+------+------------+
|area|no_of_rooms| price|    features|
+----+-----------+------+------------+
|1000|          1| 10000|[1000.0,1.0]|
|2000|          2|200000|[2000.0,2.0]|
|3000|          3|300000|[3000.0,3.0]|
|4000|          4|400000|[4000.0,4.0]|
|5000|          5|500000|[5000.0,5.0]|
+----+-----------+------+------------+



In [0]:
updated_df=updated_df.select("features","price")
updated_df

DataFrame[features: vector, price: bigint]

In [0]:
train_df,test_df=updated_df.randomSplit([0.8,0.2],seed=42)

In [0]:
test_df.show()

+------------+------+
|    features| price|
+------------+------+
|[2000.0,2.0]|200000|
+------------+------+



In [0]:
train_df.show()

+------------+------+
|    features| price|
+------------+------+
|[1000.0,1.0]| 10000|
|[3000.0,3.0]|300000|
|[4000.0,4.0]|400000|
|[5000.0,5.0]|500000|
+------------+------+



In [0]:
regression=LinearRegression(labelCol="price")

regression_model=regression.fit(train_df)

In [0]:
regression_model

LinearRegressionModel: uid=LinearRegression_d915d540955e, numFeatures=2

In [0]:
train_results=regression_model.transform(test_df)

In [0]:
train_results.show()

+------------+------+------------------+
|    features| price|        prediction|
+------------+------+------------------+
|[2000.0,2.0]|200000|148571.42857142878|
+------------+------+------------------+



In [0]:
test_results=regression_model.transform(test_df)

In [0]:
test_results=regression_model.transform(test_df)

In [0]:
test_results.show()

+------------+------+------------------+
|    features| price|        prediction|
+------------+------+------------------+
|[2000.0,2.0]|200000|148571.42857142878|
+------------+------+------------------+



In [0]:
evaluation=RegressionEvaluator(
    labelCol="price",
    predictionCol="prediction",
    metricName='r2'
)
# evaluation.evaluate(train_results)

In [0]:
evaluation.evaluate(train_results)

-inf